# TASS 2018: análisis de sentimiento con Bag-of-Words y regresión softmax

**Curso:** Procesamiento de Lenguaje Natural · Unidad 3 — Clasificación de texto  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 4 — Naive Bayes, Text Classification, and Sentiment; Cap. 5 — Logistic Regression  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook construye un **clasificador de sentimiento** para tweets en español (corpus TASS 2018, cuatro clases: positivo, negativo, neutral y sin sentimiento). Los tweets se normalizan, se representan como **Bag-of-Words con n-gramas** (palabras sueltas, pares y tríos) y se clasifican con **regresión logística multinomial (softmax)**. Como las clases están desbalanceadas, se aplica sobremuestreo de las clases minoritarias, y el modelo se evalúa con validación cruzada y en un conjunto de prueba aparte.

**Objetivo:** obtener una **línea base** sólida y reproducible para el análisis de sentimiento en español, con un protocolo de evaluación sin fuga de datos, contra la cual comparar modelos más complejos (TF-IDF con rasgos léxicos, BERT, LSTM).

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Finanzas y servicio al cliente — Menciones de una marca bancaria

- **Problema:** el banco necesita responder primero a los clientes molestos en redes sociales.
- **Entrada:** tweets que mencionan la marca.
- **Cómo se aplica BoW con n-gramas + softmax:**
  1. Los tweets se normalizan y se representan con BoW de 1 a 3 palabras, que capturan negaciones (`no me han`, `nunca responden`).
  2. La regresión logística multinomial (softmax) da la probabilidad de cada clase: P, N, NEU y NONE.
  3. El sobremuestreo se aplica dentro de cada partición de la validación cruzada para no inflar las métricas.
- **Salida:** clase de cada tweet y su probabilidad de ser negativo.
- **Decisión:** los tweets se ordenan por P(N) y los de mayor probabilidad pasan al equipo de atención.

### Turismo y hospitalidad — Reputación de un destino

- **Problema:** una oficina de turismo quiere detectar a tiempo picos de quejas sobre la ciudad.
- **Entrada:** tweets que mencionan Cartagena.
- **Cómo se aplica BoW con n-gramas + softmax:**
  1. Mismo modelo: BoW con n-gramas (`muy caro`, `no volvería`) y regresión softmax de 4 clases.
  2. La clase NONE separa los mensajes informativos (`hoy llega el crucero`) de las opiniones.
  3. Se calcula cada día la proporción de tweets negativos entre los que tienen opinión.
- **Salida:** serie diaria del porcentaje de tweets negativos.
- **Decisión:** alerta cuando la proporción de negativos supera su promedio histórico.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install numpy pandas scikit-learn imbalanced-learn matplotlib seaborn nltk spacy requests
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `pathlib.Path, collections.Counter` | Rutas y conteo de clases |
| `numpy, pandas` | Cálculo y tablas |
| `matplotlib, seaborn` | Gráficas |
| `sklearn` | `CountVectorizer`, `LogisticRegression`, métricas, `ShuffleSplit`, `LabelEncoder` |
| `imblearn.over_sampling.RandomOverSampler` | Sobremuestreo de clases minoritarias |
| `logic.text_processing.TextProcessing` | Normalizador de tweets del curso (carpeta `logic/`) |

**Datos:** `data/tass/tass2018_es_train.csv` y `tass2018_es_test.csv` (incluidos en el repositorio).

## Marco metodológico: CRISP-ML(Q)

Este notebook sigue **CRISP-ML(Q)** (*Cross-Industry Standard Process for the development of Machine Learning applications with Quality assurance*; Studer et al., 2021). El proceso tiene seis fases, y en cada una se identifican los riesgos de calidad y cómo se mitigan. Las fases aparecen marcadas con 🔷 a lo largo del notebook.

| Fase CRISP-ML(Q) | Dónde se aborda en este notebook | Aseguramiento de calidad |
|------------------|----------------------------------|--------------------------|
| 1. Entendimiento del negocio y de los datos | Secciones 2 y 4: descripción de TASS, carga y distribución de clases | Se detecta el desbalance de clases antes de modelar |
| 2. Ingeniería de datos | Secciones 5–6: normalización de tweets y BoW con n-gramas | El vectorizador se ajusta **solo con train**, para no filtrar vocabulario de prueba |
| 3. Ingeniería del modelo | Secciones 8–11: validación cruzada, sobremuestreo y regresión softmax | El sobremuestreo se aplica **dentro de cada fold**, solo a la parte de entrenamiento |
| 4. Aseguramiento de calidad del modelo | Secciones 12–13: métricas macro en CV y evaluación única en test | Métricas macro (cada clase pesa igual); test usado una sola vez |
| 5. Despliegue | Al final (fuera de alcance) | — |
| 6. Monitoreo y mantenimiento | Al final (fuera de alcance) | — |

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | El desbalance de clases es una fuente de sesgo: se mide, se corrige con sobremuestreo y se reportan métricas macro para que las clases minoritarias cuenten igual. |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | BoW + regresión logística es un modelo **interpretable**: cada coeficiente se asocia a un n-grama concreto. |
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Riesgo de **fuga de datos** identificado y mitigado: el sobremuestreo antes de la validación cruzada copiaría tweets entre entrenamiento y validación. |
| **ISO/IEC 42001:2023** — Sistema de gestión de IA en una organización | El protocolo (semillas fijas, train/test separados, métricas definidas) deja trazabilidad para auditar el modelo. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | Código y librerías abiertas; el corpus TASS se distribuye para investigación, con condiciones de uso propias. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. Cargar y preprocesar el corpus de sentimiento en tweets **TASS 2018**
2. Aplicar un **pipeline de normalización de texto** antes de vectorizar
3. Representar los tweets como vectores **Bag-of-Words con n-gramas** (`CountVectorizer`)
4. Manejar el **desbalance de clases** con `RandomOverSampler` (imbalanced-learn) **sin fuga de datos**
5. Evaluar una **regresión logística multinomial (softmax)** con validación cruzada `ShuffleSplit` de 10 particiones
6. Reportar precisión, *recall*, F1 y *accuracy* promediados entre particiones

---

## 1. Configuración del entorno

In [1]:
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import ShuffleSplit
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler

# Normalizador de tweets del curso (carpeta logic/ junto a este notebook)
from logic.text_processing import TextProcessing

DIR_DATA = Path.cwd() / 'data' / 'tass'
print(f'Directorio de datos: {DIR_DATA}')

Directorio de datos: C:\Users\epuerta\OneDrive - Universidad Tecnológica de Bolívar\Apps\courseNLP\examples\data\tass


> 🔷 **CRISP-ML(Q) · Fase 1 — Entendimiento del negocio y de los datos.** Qué datos hay, qué significa cada etiqueta y cómo se distribuyen las clases.

---

## 2. El conjunto de datos TASS 2018

[TASS](http://tass.sepln.org/) (Taller de Análisis Semántico en la SEPLN) es una campaña de evaluación de análisis de sentimiento en español. La edición 2018 usa tweets etiquetados con cuatro clases de polaridad.

| Etiqueta | Significado |
|---|---|
| **P** | Sentimiento positivo |
| **N** | Sentimiento negativo |
| **NEU** | Neutral: el tweet expresa a la vez polaridad positiva y negativa |
| **NONE** | Sin sentimiento: el tweet es informativo u objetivo |

---

## 3. Inicializar las utilidades de preprocesamiento

In [2]:
tp = TextProcessing()   # normalizador de tweets: minúsculas, URL, menciones, hashtags, emojis, puntuación
le = LabelEncoder()     # convierte las etiquetas de texto (P, N, NEU, NONE) en enteros

---

## 4. Cargar los conjuntos de entrenamiento y prueba

In [3]:
# Conjuntos oficiales de TASS 2018
data_train = pd.read_csv(DIR_DATA / 'tass2018_es_train.csv', sep=',')
data_test = pd.read_csv(DIR_DATA / 'tass2018_es_test.csv', sep=',')

print(f'Ejemplos de entrenamiento: {len(data_train):,}')
print(f'Ejemplos de prueba:        {len(data_test):,}')
print(f'\nColumnas: {data_train.columns.tolist()}')
print(data_train.head(3).to_string())

Ejemplos de entrenamiento: 1,008
Ejemplos de prueba:        506

Columnas: ['tweetid', 'user', 'content', 'date', 'lang', 'sentiment/polarity/value']
              tweetid          user                                                                                                                                          content                 date lang sentiment/polarity/value
0  768213876278165504  OnceBukowski  -Me caes muy bien \r\n-Tienes que jugar más partidas al lol con Russel y conmigo\r\n-Por qué tan Otako, deja de ser otako\r\n-Haber si me muero  2016-08-23 22:30:35   es                     NONE
1  768213567418036224      anahorxn                    @myendlesshazza a. que puto mal escribo\r\n\r\nb. me sigo surrando help \r\n\r\n3. ha quedado raro el "cómetelo" ahí JAJAJAJA  2016-08-23 22:29:21   es                        N
2  768212591105703936  martitarey13                                                @estherct209 jajajaja la tuya y la d mucha gente seguro!! Pero yo no pu

> **Interpretación de la salida:** el CSV de entrenamiento contiene el texto del tweet (`content`) y su etiqueta de polaridad (`sentiment/polarity/value`). Revisa la distribución de clases antes de modelar: si una clase domina, el clasificador tenderá a predecirla si no se corrige el desbalance.

> 🔷 **CRISP-ML(Q) · Fase 2 — Ingeniería de datos.** Normalización de tweets y conversión a vectores BoW, ajustando todo solo con entrenamiento.

---

## 5. Preprocesamiento del texto

`TextProcessing.transformer()` normaliza cada tweet: elimina tildes, pasa a minúsculas, reemplaza emojis, URL, menciones (@) y hashtags (#) por marcadores (que quedan como las palabras `emoji`, `url`, `mention` y `hastag`, porque el paso siguiente elimina los corchetes), elimina la mayoría de signos de puntuación y colapsa los espacios. Esto reduce el tamaño del vocabulario y mejora la generalización. La celda imprime un tweet antes y después para comprobarlo.

In [4]:
# Aplica el normalizador del curso a cada tweet
x_train = [tp.transformer(row) or '' for row in data_train['content'].tolist()]
y_train = le.fit_transform(data_train['sentiment/polarity/value'].tolist())

x_test = [tp.transformer(row) or '' for row in data_test['content'].tolist()]
y_test = le.transform(data_test['sentiment/polarity/value'].tolist())

print(f'Clases: {le.classes_.tolist()}')
print(f'Distribución de clases en entrenamiento: {sorted(Counter(y_train).items())}')
print(f'\nEjemplo original:    {data_train["content"].iloc[1]!r}')
print(f'Ejemplo normalizado: {x_train[1]!r}')

Clases: ['N', 'NEU', 'NONE', 'P']
Distribución de clases en entrenamiento: [(np.int64(0), 418), (np.int64(1), 133), (np.int64(2), 139), (np.int64(3), 318)]

Ejemplo original:    '@myendlesshazza a. que puto mal escribo\r\n\r\nb. me sigo surrando help \r\n\r\n3. ha quedado raro el "cómetelo" ahí JAJAJAJA'
Ejemplo normalizado: 'mention a. que puto mal escribo b. me sigo surrando help 3. ha quedado raro el cometelo ahi jajajaja'


> **Interpretación de la salida:** `LabelEncoder` asigna los enteros en orden alfabético (N=0, NEU=1, NONE=2, P=3). La distribución muestra el desbalance típico de los datos de Twitter: N y P dominan, mientras que NEU y NONE son minoritarias. El sobremuestreo de la sección 9 lo compensa.

---

## 6. Vectorización Bag-of-Words con n-gramas

`CountVectorizer(ngram_range=(1, 3))` construye un vocabulario de unigramas, bigramas y trigramas. Los bigramas y trigramas capturan patrones de orden local que los unigramas pierden (por ejemplo, *“no me gusta”* frente a *“me gusta”*).

In [5]:
# El vectorizador se ajusta solo con entrenamiento para evitar fuga de datos
bow = CountVectorizer(analyzer='word', ngram_range=(1, 3))
x_train = bow.fit_transform(x_train)
x_test = bow.transform(x_test)

print(f'Tamaño del vocabulario: {len(bow.vocabulary_):,}')
print(f'Matriz de entrenamiento: {x_train.shape}')
print(f'Matriz de prueba:        {x_test.shape}')
density = x_train.nnz / (x_train.shape[0] * x_train.shape[1])
print(f'Dispersión de la matriz: {1 - density:.4%}')

Tamaño del vocabulario: 25,996
Matriz de entrenamiento: (1008, 25996)
Matriz de prueba:        (506, 25996)
Dispersión de la matriz: 99.8502%


> **Interpretación de la salida:** incluir trigramas amplía mucho el vocabulario. La dispersión es altísima porque cada tweet es corto (unas 15 palabras de mediana) mientras que el vocabulario combinado tiene miles de n-gramas. `CountVectorizer` devuelve una matriz CSR (*Compressed Sparse Row*) que almacena solo los valores distintos de cero.

---

## 7. Distribución de clases

In [6]:
# Número de tweets por clase (0=N, 1=NEU, 2=NONE, 3=P)
print('Distribución en entrenamiento:', sorted(Counter(y_train).items()))
print('Distribución en prueba:       ', sorted(Counter(y_test).items()))

Distribución en entrenamiento: [(np.int64(0), 418), (np.int64(1), 133), (np.int64(2), 139), (np.int64(3), 318)]
Distribución en prueba:        [(np.int64(0), 219), (np.int64(1), 69), (np.int64(2), 62), (np.int64(3), 156)]


> 🔷 **CRISP-ML(Q) · Fase 3 — Ingeniería del modelo.** Protocolo de validación, corrección del desbalance y entrenamiento del clasificador.

---

## 8. Configuración de la validación cruzada — ShuffleSplit

`ShuffleSplit` baraja y divide los datos de entrenamiento `n_splits` veces, y en cada partición evalúa el modelo en un 25 % reservado. A diferencia de `KFold`, cada partición es aleatoria e independiente, así que los conjuntos de validación pueden solaparse entre particiones.

In [7]:
# 10 particiones aleatorias 75/25; random_state garantiza la reproducibilidad
k_fold = ShuffleSplit(n_splits=10, test_size=0.25, random_state=42)

---

## 9. Manejo del desbalance — RandomOverSampler

`RandomOverSampler` duplica ejemplos de las clases minoritarias hasta que todas tienen tantos como la mayoritaria.

> ⚠️ **Dónde aplicarlo importa.** Si se sobremuestrea **antes** de la validación cruzada, copias del mismo tweet quedan en entrenamiento y en validación, y las métricas salen infladas (fuga de datos). Por eso, en la sección 11 el sobremuestreo se aplica **dentro de cada partición, solo a la parte de entrenamiento**. Tampoco se sobremuestrea el conjunto de prueba: debe reflejar la distribución real.

La celda siguiente sobremuestrea todo el conjunto de entrenamiento. Ese resultado se usa **solo para el modelo final** (sección 13).

In [8]:
# Sobremuestreo de todo train: solo para entrenar el modelo final de la sección 13
ros = RandomOverSampler(random_state=1000)
x_train_balanced, y_train_balanced = ros.fit_resample(x_train, y_train)
print('Después del sobremuestreo (train completo):', sorted(Counter(y_train_balanced).items()))

Después del sobremuestreo (train completo): [(np.int64(0), 418), (np.int64(1), 418), (np.int64(2), 418), (np.int64(3), 418)]


> **Interpretación de la salida:** tras el sobremuestreo, las cuatro clases tienen el mismo número de ejemplos. Esto evita que el clasificador favorezca a la clase mayoritaria. Ojo: duplicar ejemplos no añade información nueva; SMOTE (*Synthetic Minority Oversampling Technique*) genera ejemplos sintéticos interpolando, aunque con texto disperso su beneficio es limitado.

---

## 10. Clasificador — regresión logística multinomial (softmax)

Con el *solver* `lbfgs` y más de dos clases, `LogisticRegression` optimiza directamente la **función softmax** sobre todas las clases a la vez (no uno contra el resto). `C=10` aplica poca regularización, lo que permite al modelo ajustarse mucho a los datos de entrenamiento.

In [9]:
# C=10 implica poca regularización L2; redúcelo (p. ej. C=0.1) si el modelo se sobreajusta
softmax = LogisticRegression(solver='lbfgs', C=10, max_iter=1000)

---

## 11. Entrenamiento con validación cruzada (10 particiones)

En cada partición: (1) se separan entrenamiento y validación del conjunto **original**; (2) se sobremuestrea **solo** la parte de entrenamiento; (3) se entrena y se evalúa en la validación sin tocar.

In [10]:
accuracy_scores = []
recall_scores = []
precision_scores = []
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(k_fold.split(x_train, y_train)):
    # Partición sobre los datos ORIGINALES (sin duplicados)
    x_fold_train, y_fold_train = x_train[train_idx], y_train[train_idx]
    x_fold_val, y_fold_val = x_train[val_idx], y_train[val_idx]

    # Sobremuestreo solo de la parte de entrenamiento de esta partición
    x_fold_train, y_fold_train = RandomOverSampler(random_state=1000).fit_resample(x_fold_train, y_fold_train)

    softmax.fit(x_fold_train, y_fold_train)
    y_pred_val = softmax.predict(x_fold_val)

    # Métricas macro: cada clase pesa lo mismo
    accuracy_scores.append(accuracy_score(y_fold_val, y_pred_val))
    recall_scores.append(recall_score(y_fold_val, y_pred_val, average='macro', zero_division=0))
    precision_scores.append(precision_score(y_fold_val, y_pred_val, average='macro', zero_division=0))
    f1_scores.append(f1_score(y_fold_val, y_pred_val, average='macro', zero_division=0))

print('Validación cruzada completa.')

Validación cruzada completa.


> 🔷 **CRISP-ML(Q) · Fase 4 — Aseguramiento de calidad del modelo.** Métricas macro en validación cruzada y una única evaluación en prueba.

---

## 12. Métricas promedio de validación

In [11]:
# Promedio de las 10 particiones, en porcentaje
avg_accuracy = round(np.mean(accuracy_scores) * 100, 2)
avg_recall = round(np.mean(recall_scores) * 100, 2)
avg_precision = round(np.mean(precision_scores) * 100, 2)
avg_f1 = round(np.mean(f1_scores) * 100, 2)

print(f'Promedio de {k_fold.n_splits} particiones (promedio macro):')
print(f'  Accuracy:  {avg_accuracy:.2f}% ± {np.std(accuracy_scores) * 100:.2f}')
print(f'  Precisión: {avg_precision:.2f}% ± {np.std(precision_scores) * 100:.2f}')
print(f'  Recall:    {avg_recall:.2f}% ± {np.std(recall_scores) * 100:.2f}')
print(f'  F1:        {avg_f1:.2f}% ± {np.std(f1_scores) * 100:.2f}')

Promedio de 10 particiones (promedio macro):
  Accuracy:  48.73% ± 2.35
  Precisión: 39.30% ± 4.34
  Recall:    36.05% ± 2.08
  F1:        34.71% ± 2.71


> **Interpretación de la salida:** las métricas **macro** dan el mismo peso a cada clase sin importar cuántos ejemplos tenga, lo adecuado con clases desbalanceadas. La validación se hace sobre datos **no sobremuestreados**, así que refleja la distribución real. La desviación indica cuánto varía el resultado entre particiones.
>
> **Efecto de la fuga de datos:** la versión anterior de este notebook sobremuestreaba **antes** de la validación cruzada y reportaba un F1 macro de validación cercano al **80 %**, frente a ~35 % en prueba. Con el sobremuestreo dentro de cada partición, la validación (~35 %) coincide con la prueba: esa es la estimación honesta.
>
> Para situar la cifra, compárala con los sistemas participantes en las actas de TASS 2018 (SEPLN).

---

## 13. Evaluación final en el conjunto de prueba

In [12]:
# Entrena con todo el entrenamiento sobremuestreado y evalúa una sola vez en prueba
softmax.fit(x_train_balanced, y_train_balanced)
y_pred_test = softmax.predict(x_test)

test_f1 = round(f1_score(y_test, y_pred_test, average='macro', zero_division=0) * 100, 2)
test_acc = round(accuracy_score(y_test, y_pred_test) * 100, 2)
test_precision = round(precision_score(y_test, y_pred_test, average='macro', zero_division=0) * 100, 2)
test_recall = round(recall_score(y_test, y_pred_test, average='macro', zero_division=0) * 100, 2)

results = {
    'F1': test_f1,
    'Accuracy': test_acc,
    'Precisión': test_precision,
    'Recall': test_recall,
}
for metric, value in results.items():
    print(f'{metric}: {value}%')

F1: 37.56%
Accuracy: 51.78%
Precisión: 40.31%
Recall: 37.98%


> **Interpretación de la salida:** las métricas de prueba miden la capacidad de generalización real. Compáralas con los promedios de la validación cruzada: si la validación cruzada es mucho mayor que la prueba, el modelo se sobreajusta a la distribución de entrenamiento. En ese caso, reduce `C` (más regularización) o acorta el rango de n-gramas.

---

## 14. Extensiones sugeridas

1. **TF-IDF en lugar de CountVectorizer:** reemplaza los conteos por pesos normalizados (ver `tfidf_lexical_tass.ipynb`).
2. **Embeddings en español:** sustituir BoW por FastText (`cc.es.300.bin`) o por un BERT en español (ver `bert_classifier.ipynb`).
3. **SMOTE:** genera ejemplos sintéticos de las clases minoritarias interpolando, en lugar de duplicarlos.
4. **Análisis de errores:** revisar los tweets mal clasificados por clase; NEU y NONE suelen ser las más difíciles de separar.
5. **Adaptación de dominio:** reentrenar con datos del dominio de destino (reseñas de hoteles, tweets políticos) para mejorar el desempeño en esa aplicación.

---

> 🔷 **CRISP-ML(Q) · Fases 5 y 6 — Despliegue, monitoreo y mantenimiento.** Están fuera del alcance de este notebook académico, pero así se abordarían:
>
> - **Despliegue:** guardar el vectorizador y el modelo juntos en un `Pipeline` con `joblib`, y exponerlo como servicio que reciba un tweet y devuelva su polaridad.
> - **Monitoreo y mantenimiento:** registrar la distribución de clases predichas y el porcentaje de n-gramas desconocidos por semana; si cambian mucho (nuevos temas, jerga), reetiquetar una muestra y reentrenar.